## Lossless Audio Compression Using Rice Coding

This project implements a lossless audio compression and decompression system using Differential Pulse Code Modulation (DPCM), ZigZag mapping and Rice coding.

The implementation evaluates two Rice parameters, K=2 and K=4, using two audio samples to compare compression efficiency and verify lossless reconstruction.

## 1. Libraries and Configuration

In [1]:
import os
import math
from pathlib import Path
import numpy as np
from scipy.io import wavfile

## 2. Audio Loading and Preprocessing

In [2]:
# Load sound1.wav and sound2.wav into SOUND_FILES
DATA_DIR = Path("audio")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOUND_FILES = ["Sound1.wav", "Sound2.wav"]
K_VALUES = [2, 4]

# keeps samples as integers to preserve lossless behaviour
def read_wav_int(path: Path):
    sr, data = wavfile.read(str(path))
    return sr, data

def file_size_bytes(path: Path) -> int:
    return path.stat().st_size

def ensure_mono_or_handle_stereo(samples: np.ndarray) -> np.ndarray:
    return samples

# Quick sanity check: list available WAVs
for f in SOUND_FILES:
    p = DATA_DIR / f
    print(f"{f}: {'FOUND' if p.exists() else 'MISSING'} -> {p}")


Sound1.wav: FOUND -> audio\Sound1.wav
Sound2.wav: FOUND -> audio\Sound2.wav


In [3]:
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

The files are read in integer form so that the original sample values remain unchanged. For each WAV file, the code records metadata such as sample rate, channel count, data type, number of samples, duration, and original file size.

In [4]:
# Exercise 2 — Load WAV files

audio_data = {}

for fname in SOUND_FILES:
    path = DATA_DIR / fname
    sr, samples = read_wav_int(path)
    samples = ensure_mono_or_handle_stereo(samples)

    duration_sec = samples.shape[0] / sr
    channels = 1 if samples.ndim == 1 else samples.shape[1]

    audio_data[fname] = {
        "sample_rate": sr,
        "samples": samples,
        "channels": channels,
        "dtype": samples.dtype,
        "num_samples": samples.shape[0],
        "duration_sec": duration_sec,
        "file_size_bytes": file_size_bytes(path),
    }




Before Rice coding is applied, the raw audio samples are converted into residual values using first-order DPCM. In this approach, each sample is predicted from the previous sample, and the difference between the actual sample and the predicted one is stored as the residual.

## 3. DPCM and ZigZag Mapping

In [7]:
# Residuals + ZigZag + Rice Encoding (K=2/4)
# ---------- Residuals (DPCM) ----------
def compute_dpcm_residuals(samples: np.ndarray) -> np.ndarray:
    x = samples.astype(np.int32)

    if x.ndim == 1:
        pred = np.empty_like(x)
        pred[0] = 0
        pred[1:] = x[:-1]
        return (x - pred).astype(np.int32)

    if x.ndim == 2:
        pred = np.empty_like(x)
        pred[0, :] = 0
        pred[1:, :] = x[:-1, :]
        return (x - pred).astype(np.int32)

    raise ValueError(f"Unsupported samples shape: {x.shape}")


# ---------- ZigZag mapping (signed -> unsigned) ----------
#The DPCM residuals produced in the previous stage can contain both positive and negative values. 
#Since Rice coding is more naturally applied to non-negative integers, the residuals are first transformed using ZigZag mapping.
def zigzag_encode_signed_to_unsigned(x: np.ndarray) -> np.ndarray:
    """
    Zigzag mapping:
      0 -> 0
      -1 -> 1
      +1 -> 2
      -2 -> 3
      +2 -> 4
      ...

    Returns uint32, same shape as input.
    """
    x = x.astype(np.int32)
    u = (np.abs(x) << 1) - (x > 0).astype(np.int32)  # abs(x)*2 - 1 if positive else abs(x)*2
    return u.astype(np.uint32)

 
# ---------- Bit writer ----------
#Rice coding produces output at the bit level rather than directly as full bytes. 
#To store the encoded data efficiently, a helper class is used to accumulate individual bits and pack them into bytes in MSB-first order.
class BitWriter:
    """Pack bits into bytes (MSB-first)."""
    def __init__(self):
        self._bytes = bytearray()
        self._cur = 0
        self._nbits = 0

    def write_bit(self, b: int):
        self._cur = (self._cur << 1) | (1 if b else 0)
        self._nbits += 1
        if self._nbits == 8:
            self._bytes.append(self._cur & 0xFF)
            self._cur = 0
            self._nbits = 0

    def write_bits(self, value: int, n: int):
        for i in reversed(range(n)):
            self.write_bit((value >> i) & 1)

    def get_bytes(self) -> bytes:
        if self._nbits > 0:
            self._cur <<= (8 - self._nbits)
            self._bytes.append(self._cur & 0xFF)
            self._cur = 0
            self._nbits = 0
        return bytes(self._bytes)


# ---------- Rice encoding (unsigned ints) ----------
#he main compression stage applies Rice encoding to the unsigned values generated after ZigZag mapping. For each value, the algorithm separates it into two parts: a quotient and a remainder. 
#The quotient is written in unary form, while the remainder is written using a fixed number of bits determined by the parameter 𝐾
def rice_encode_unsigned(u: np.ndarray, K: int) -> bytes:
    """
    Rice code for each value:
      q = u >> K
      r = u & ((1<<K)-1)
      unary(q): q zeros then 1
      then K-bit remainder r
    """
    u = u.astype(np.uint32).ravel()
    bw = BitWriter()
    mask = (1 << K) - 1

    for val in u:
        q = int(val >> K)
        r = int(val & mask)

        # unary(q): q zeros then 1
        for _ in range(q):
            bw.write_bit(0)
        bw.write_bit(1)

        # remainder
        if K > 0:
            bw.write_bits(r, K)

    return bw.get_bytes()


# ---------- Write .ex2 ----------
def write_ex2_file(out_path: Path, payload: bytes, meta: dict):
    """
    Save header (ASCII lines) + blank line + payload bytes.
    Header is for convenience/debugging (not required by spec, but helpful).
    """
    header_lines = [
        "EX2_RICE_V1",
        f"K={meta['K']}",
        f"sample_rate={meta['sample_rate']}",
        f"dtype={meta['dtype']}",
        f"channels={meta['channels']}",
        f"num_samples={meta['num_samples']}",
    ]
    header = ("\n".join(header_lines) + "\n\n").encode("utf-8")

    with open(out_path, "wb") as f:
        f.write(header)
        f.write(payload)


# ---------- Encode WAV in audio_data ----------
def encode_wav_to_ex2(fname: str, K: int) -> Path:
    info = audio_data[fname]

    # Ensure residuals exist (compute if missing)
    if "residuals" not in info:
        info["residuals"] = compute_dpcm_residuals(info["samples"])
        r = info["residuals"]
        info["residual_min"] = int(r.min())
        info["residual_max"] = int(r.max())
        info["residual_mean_abs"] = float(np.mean(np.abs(r)))

    residuals = info["residuals"]
    u = zigzag_encode_signed_to_unsigned(residuals)

    payload = rice_encode_unsigned(u, K)

    out_name = Path(fname).stem + f"_K{K}_Enc.ex2"
    out_path = OUTPUT_DIR / out_name

    meta = {
        "K": K,
        "sample_rate": info["sample_rate"],
        "dtype": str(info["dtype"]),
        "channels": info["channels"],
        "num_samples": info["num_samples"],
    }
    write_ex2_file(out_path, payload, meta)
    return out_path


# ---------- Run for all files and K ----------
encoded_paths = {}

for fname in SOUND_FILES:
    orig_size = audio_data[fname]["file_size_bytes"]
    encoded_paths[fname] = {}

    # Print residual stats (computed if not present)
    if "residuals" not in audio_data[fname]:
        _ = compute_dpcm_residuals(audio_data[fname]["samples"])
        audio_data[fname]["residuals"] = _
    r = audio_data[fname]["residuals"]
    print(f"{fname} residuals: min={int(r.min())}, max={int(r.max())}, mean|r|={float(np.mean(np.abs(r))):.2f}")

    for K in K_VALUES:
        out_path = encode_wav_to_ex2(fname, K)
        enc_size = out_path.stat().st_size
        comp_pct = (1.0 - (enc_size / orig_size)) * 100.0

        encoded_paths[fname][K] = out_path

        print(f"  K={K} -> {out_path.name}")
        print(f"    Original: {orig_size} bytes")
        print(f"    Encoded : {enc_size} bytes")
        print(f"    %Compression: {comp_pct:.2f}%")
    print("-" * 50)


Sound1.wav residuals: min=-3571, max=3469, mean|r|=72.49
  K=2 -> Sound1_K2_Enc.ex2
    Original: 1002088 bytes
    Encoded : 2429819 bytes
    %Compression: -142.48%
  K=4 -> Sound1_K4_Enc.ex2
    Original: 1002088 bytes
    Encoded : 857521 bytes
    %Compression: 14.43%
--------------------------------------------------
Sound2.wav residuals: min=-50206, max=50712, mean|r|=7175.84
  K=2 -> Sound2_K2_Enc.ex2
    Original: 1008044 bytes
    Encoded : 226196619 bytes
    %Compression: -22339.16%
  K=4 -> Sound2_K4_Enc.ex2
    Original: 1008044 bytes
    Encoded : 56793317 bytes
    %Compression: -5534.01%
--------------------------------------------------


In [8]:
# Decode and compare (verify lossless)

from scipy.io import wavfile

# ---------- Bit reader ----------
class BitReader:
    """Read bits from bytes (MSB-first)."""
    def __init__(self, data: bytes):
        self.data = data
        self.i = 0          # byte index
        self.cur = 0        # current byte
        self.nbits = 0      # bits remaining in cur

    def read_bit(self) -> int:
        if self.nbits == 0:
            if self.i >= len(self.data):
                raise EOFError("Ran out of bits while decoding.")
            self.cur = self.data[self.i]
            self.i += 1
            self.nbits = 8
        self.nbits -= 1
        return (self.cur >> self.nbits) & 1

    def read_bits(self, n: int) -> int:
        v = 0
        for _ in range(n):
            v = (v << 1) | self.read_bit()
        return v

# ---------- Inverse ZigZag (unsigned -> signed) ----------
def zigzag_decode_unsigned_to_signed(u: np.ndarray) -> np.ndarray:
    """
    Inverse of the zigzag mapping used earlier:
      u even -> negative or zero
      u odd  -> positive
    Using mapping:
      signed = -(u//2) if u even else (u+1)//2
    Returns int32.
    """
    u = u.astype(np.uint32)
    signed = np.empty_like(u, dtype=np.int32)
    even = (u % 2 == 0)
    signed[even] = -(u[even] // 2).astype(np.int32)
    signed[~even] = ((u[~even] + 1) // 2).astype(np.int32)
    return signed

# ---------- Rice decode (unsigned ints) ----------
def rice_decode_unsigned(payload: bytes, K: int, count: int) -> np.ndarray:
    """
    Decode 'count' unsigned integers from Rice-coded payload.
    For each value:
      read unary q: number of 0s until a 1
      read K-bit remainder r
      u = (q << K) | r
    """
    br = BitReader(payload)
    out = np.empty(count, dtype=np.uint32)

    for i in range(count):
        # unary quotient: count zeros until 1
        q = 0
        while True:
            b = br.read_bit()
            if b == 1:
                break
            q += 1

        r = br.read_bits(K) if K > 0 else 0
        out[i] = (q << K) | r

    return out

# ---------- Inverse DPCM (residuals -> samples) ----------
def reconstruct_from_dpcm_residuals(residuals: np.ndarray, dtype: np.dtype) -> np.ndarray:
    """
    Given residuals e[n] and predictor x[n-1], reconstruct:
      x[0] = e[0]
      x[n] = e[n] + x[n-1]
    Works for mono (1D) or stereo (2D).
    Returns array cast back to original dtype (int16/int32).
    """
    e = residuals.astype(np.int32)

    if e.ndim == 1:
        x = np.empty_like(e, dtype=np.int32)
        x[0] = e[0]
        for n in range(1, e.shape[0]):
            x[n] = e[n] + x[n-1]
        return x.astype(dtype)

    if e.ndim == 2:
        x = np.empty_like(e, dtype=np.int32)
        x[0, :] = e[0, :]
        for n in range(1, e.shape[0]):
            x[n, :] = e[n, :] + x[n-1, :]
        return x.astype(dtype)

    raise ValueError(f"Unsupported residuals shape: {e.shape}")

# ---------- Read our .ex2 file (header + payload) ----------
def read_ex2_file(path: Path):
    """
    Reads header lines until blank line, then returns (meta_dict, payload_bytes).
    Header lines look like:
      EX2_RICE_V1
      K=2
      sample_rate=...
      dtype=int16
      channels=1
      num_samples=...
    """
    with open(path, "rb") as f:
        content = f.read()

    sep = b"\n\n"
    if sep not in content:
        raise ValueError("Invalid .ex2 file: missing header separator (blank line).")

    header_bytes, payload = content.split(sep, 1)
    lines = header_bytes.decode("utf-8").splitlines()

    if not lines or lines[0].strip() != "EX2_RICE_V1":
        raise ValueError("Invalid .ex2 header (missing EX2_RICE_V1).")

    meta = {}
    for line in lines[1:]:
        if "=" in line:
            k, v = line.split("=", 1)
            meta[k.strip()] = v.strip()

    # Parse types
    meta_parsed = {
        "K": int(meta["K"]),
        "sample_rate": int(meta["sample_rate"]),
        "dtype": np.dtype(meta["dtype"]),
        "channels": int(meta["channels"]),
        "num_samples": int(meta["num_samples"]),
    }
    return meta_parsed, payload

# ---------- Decode a specific encoded file back to WAV ----------
def decode_ex2_to_wav(ex2_path: Path, out_wav_path: Path, original_samples: np.ndarray, original_sr: int):
    meta, payload = read_ex2_file(ex2_path)

    K = meta["K"]
    sr = meta["sample_rate"]
    dtype = meta["dtype"]
    channels = meta["channels"]
    num_samples = meta["num_samples"]

    # Consistency checks
    if sr != original_sr:
        print(f"WARNING: sample_rate mismatch: ex2={sr}, original={original_sr}")
    if channels != (1 if original_samples.ndim == 1 else original_samples.shape[1]):
        print(f"WARNING: channels mismatch: ex2={channels}, original={original_samples.shape}")

    # Determine how many values to decode (num_samples * channels)
    total_vals = num_samples * channels

    u_flat = rice_decode_unsigned(payload, K=K, count=total_vals)
    signed_flat = zigzag_decode_unsigned_to_signed(u_flat)

    # Reshape back to mono/stereo residual array
    if channels == 1:
        residuals = signed_flat.reshape((num_samples,))
    else:
        residuals = signed_flat.reshape((num_samples, channels))

    # Reconstruct original samples from residuals
    decoded_samples = reconstruct_from_dpcm_residuals(residuals, dtype=dtype)

    # Write decoded wav
    wavfile.write(str(out_wav_path), sr, decoded_samples)

    # Lossless verification (exact match)
    lossless = np.array_equal(original_samples, decoded_samples)
    return decoded_samples, lossless

# ---------- Run decoding for all files and K, then verify ----------
for fname in SOUND_FILES:
    orig_sr = audio_data[fname]["sample_rate"]
    orig_samples = audio_data[fname]["samples"]

    for K in K_VALUES:
        ex2_path = encoded_paths[fname][K]
        out_wav = OUTPUT_DIR / (Path(fname).stem + f"_K{K}_Enc_Dec.wav")

        decoded_samples, ok = decode_ex2_to_wav(
            ex2_path=ex2_path,
            out_wav_path=out_wav,
            original_samples=orig_samples,
            original_sr=orig_sr
        )

        print(f"{fname} | K={K}")
        print(f"  Decoded WAV: {out_wav.name}")
        print(f"  Lossless match (np.array_equal): {ok}")
        print("-" * 40)


Sound1.wav | K=2
  Decoded WAV: Sound1_K2_Enc_Dec.wav
  Lossless match (np.array_equal): True
----------------------------------------
Sound1.wav | K=4
  Decoded WAV: Sound1_K4_Enc_Dec.wav
  Lossless match (np.array_equal): True
----------------------------------------
Sound2.wav | K=2
  Decoded WAV: Sound2_K2_Enc_Dec.wav
  Lossless match (np.array_equal): True
----------------------------------------
Sound2.wav | K=4
  Decoded WAV: Sound2_K4_Enc_Dec.wav
  Lossless match (np.array_equal): True
----------------------------------------


In [9]:
import pandas as pd

rows = []

for fname in SOUND_FILES:
    orig_size = audio_data[fname]["file_size_bytes"]

    # Sizes for both K values
    k4_path = encoded_paths[fname][4]
    k2_path = encoded_paths[fname][2]

    size_k4 = k4_path.stat().st_size
    size_k2 = k2_path.stat().st_size

    comp_k4 = (1 - size_k4 / orig_size) * 100
    comp_k2 = (1 - size_k2 / orig_size) * 100

    rows.append({
        "File": fname,
        "Original size (bytes)": orig_size,
        "Rice (K = 4 bits)": size_k4,
        "Rice (K = 2 bits)": size_k2,
        "% Compression (K = 4 bits)": round(comp_k4, 2),
        "% Compression (K = 2 bits)": round(comp_k2, 2),
    })

df_results = pd.DataFrame(rows)
df_results


,File,Original size (bytes),Rice (K = 4 bits),Rice (K = 2 bits),% Compression (K = 4 bits),% Compression (K = 2 bits)
0,Sound1.wav,1002088,857521,2429819,14.43,-142.48
1,Sound2.wav,1008044,56793317,226196619,-5534.01,-22339.16


## Results and Discussion

The experiment compared Rice coding with K=2 and K=4 on two audio samples.

For Sound1, K=4 achieved approximately 14.43% compression, while K=2 resulted in approximately 142.48% file-size expansion.

For Sound2, both parameters resulted in substantial expansion, demonstrating that Rice coding is not equally effective for all audio signals.

The decoded audio was verified against the original samples using np.array_equal(), confirming lossless reconstruction.

These findings demonstrate that Rice coding efficiency depends on the distribution of the encoded values and the selection of the Rice parameter.